In [ ]:
# Importar librerías estándar para análisis de datos y visualización
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Herramientas de Scikit-learn para métricas de clasificación
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# TensorFlow / Keras: API Sequential, capas convolucionales, modelos prefabricados y utilidades
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout, Conv2D, MaxPool2D, Flatten, BatchNormalization, Resizing, GlobalAveragePooling2D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import EfficientNetB0

In [ ]:
# Dataset CIFAR-10 incluido en Keras: 50,000 train + 10,000 test
from tensorflow.keras.datasets import cifar10

In [ ]:
# Fijar semillas para reproducibilidad de resultados
np.random.seed(42)
tf.random.set_seed(42)

### Análisis Exploratorio y Visualización

In [ ]:
# Cargar el dataset CIFAR-10
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

In [ ]:
# Aplanar etiquetas de forma (N, 1) a (N,) para compatibilidad con sparse_categorical_crossentropy
y_train = y_train.flatten()
y_test = y_test.flatten()

In [ ]:
# Forma del conjunto de entrenamiento: (muestras, alto, ancho, canales RGB)
X_train.shape

In [ ]:
# Extraer una única imagen para inspeccionarla
single_image = X_train[0]

In [ ]:
# Dimensiones de una imagen individual: 32x32 píxeles en color (3 canales RGB)
single_image.shape

In [ ]:
# Visualizar la imagen seleccionada (CIFAR-10 es a color)
plt.imshow(single_image);

### Etiquetado

In [ ]:
# Nombres de las 10 clases de CIFAR-10
class_names = [
    'airplane', 'automobile', 'bird', 'cat', 'deer', 
    'dog', 'frog', 'horse', 'ship', 'truck'
]

# Valores de las etiquetas en el conjunto de entrenamiento (enteros del 0 al 9)
y_train[:10]

In [ ]:
# Forma del vector de etiquetas tras el flatten
y_train.shape

### Escalado

In [ ]:
# Valor máximo de los píxeles en CIFAR-10 (imágenes RGB: 0-255)
X_train.max()

In [ ]:
# Valor mínimo de los píxeles
X_train.min()

In [ ]:
# Crear dos versiones de los datos para los dos modelos:
# 1) Rango [0, 255] como float32: requerido por EfficientNetB0 (normaliza internamente)
# 2) Rango [0, 1]: estándar para CNNs entrenadas desde cero
X_train_raw = X_train.astype('float32')
X_test_raw = X_test.astype('float32')

X_train_norm = X_train.astype('float32') / 255.0
X_test_norm = X_test.astype('float32') / 255.0

In [ ]:
# Forma del conjunto de entrenamiento normalizado (usado por CNN custom)
X_train_norm.shape

In [ ]:
# Forma del conjunto de prueba normalizado (usado por CNN custom)
X_test_norm.shape

### Modelo 1: CNN Custom

In [ ]:
# Definir arquitectura de CNN moderna diseñada específicamente para CIFAR-10 (32x32x3)
model_custom = Sequential([
    Input(shape=(32, 32, 3)),
    
    # Bloque convolucional 1: extracción de características de bajo nivel (bordes, texturas)
    Conv2D(filters=32, kernel_size=(3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(filters=32, kernel_size=(3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPool2D(pool_size=(2, 2)),
    Dropout(0.2),
    
    # Bloque convolucional 2: extracción de características de nivel medio
    Conv2D(filters=64, kernel_size=(3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(filters=64, kernel_size=(3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPool2D(pool_size=(2, 2)),
    Dropout(0.3),
    
    # Bloque convolucional 3: extracción de características de alto nivel
    Conv2D(filters=128, kernel_size=(3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(filters=128, kernel_size=(3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPool2D(pool_size=(2, 2)),
    Dropout(0.4),
    
    # Aplanar la salida 2D en un vector 1D para las capas densas
    Flatten(),
    
    # Capa densa oculta con alta capacidad de representación
    Dense(units=256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    
    # Capa de salida: 10 neuronas (una por clase) con softmax para probabilidades
    Dense(units=10, activation='softmax')
])

In [ ]:
# Compilar el modelo: sparse_categorical_crossentropy para etiquetas enteras
model_custom.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
# EarlyStopping y ReduceLROnPlateau para optimizar el entrenamiento
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
lr_reduce = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

In [ ]:
# Entrenar el modelo custom con callbacks
history_custom = model_custom.fit(
    x=X_train_norm,
    y=y_train,
    validation_data=(X_test_norm, y_test),
    callbacks=[early_stop, lr_reduce],
    batch_size=128,
    epochs=50
)

#### Evaluación del Modelo Custom

In [ ]:
# Convertir el historial de entrenamiento a DataFrame para graficar
history_df = pd.DataFrame(history_custom.history)

In [ ]:
# Graficar la evolución de la accuracy en entrenamiento y validación
history_df[['accuracy', 'val_accuracy']].plot();

In [ ]:
# Graficar la evolución de la pérdida en entrenamiento y validación
history_df[['loss', 'val_loss']].plot();

In [ ]:
# Evaluar el modelo en el conjunto de prueba
loss, accuracy = model_custom.evaluate(X_test_norm, y_test, verbose=0)
print(f'Loss en test: {loss:.4f}')
print(f'Accuracy en test: {accuracy:.4f}')

In [ ]:
# Predecir clases para todo el conjunto de prueba normalizado
predictions_custom = np.argmax(model_custom.predict(X_test_norm), axis=1)

In [ ]:
# Reporte de clasificación por clase
print(classification_report(y_test, predictions_custom, target_names=class_names))

In [ ]:
# Matriz de confusión del modelo custom
plt.figure(figsize=(10, 8))
sns.heatmap(
    confusion_matrix(y_test, predictions_custom), 
    annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names
)
plt.xlabel('Predicción')
plt.ylabel('Real')

### Modelo 2: Transfer Learning con EfficientNetB0

EfficientNetB0 es un modelo pre-entrenado en ImageNet que espera imágenes de 224×224 píxeles.
El enfoque de **Transfer Learning** consiste en:
1. Cargar los pesos pre-entrenados en ImageNet (`weights='imagenet'`).
2. Congelar las capas base para que no se re-entrenen inicialmente.
3. Agregar una 'cabeza' (head) personalizada para clasificar en 10 clases de CIFAR-10.
4. Entrenar solo la cabeza sobre CIFAR-10.

In [ ]:
# Cargar EfficientNetB0 pre-entrenado en ImageNet SIN la cabeza clasificadora
# include_top=False remueve la capa densa final de 1000 clases de ImageNet
base_model = EfficientNetB0(
    include_top=False, 
    weights='imagenet',        # Usar pesos pre-entrenados (descarga automática la primera vez)
    input_shape=(224, 224, 3)  # Tamaño de entrada que espera EfficientNetB0
)

# Congelar todas las capas del modelo base para entrenar solo la cabeza personalizada
base_model.trainable = False

In [ ]:
# Construir el modelo completo con transfer learning
model_transfer = Sequential([
    Input(shape=(32, 32, 3)),
    
    # Redimensionar imágenes de 32x32 a 224x224 para EfficientNetB0
    # interpolation='bilinear' es el método estándar de redimensionamiento
    Resizing(224, 224, interpolation='bilinear'),
    
    # Modelo base pre-entrenado (congelado en esta etapa)
    base_model,
    
    # Global Average Pooling: reduce cada mapa de características a un solo valor
    GlobalAveragePooling2D(),
    
    # Cabeza personalizada con regularización para CIFAR-10
    BatchNormalization(),
    Dropout(0.5),
    Dense(units=256, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    
    # Capa de salida: 10 clases con softmax
    Dense(units=10, activation='softmax')
])

In [ ]:
# Compilar el modelo de transfer learning
model_transfer.compile(
    loss='sparse_categorical_crossentropy', 
    optimizer='adam', 
    metrics=['accuracy']
)

In [ ]:
# EarlyStopping para detener si la pérdida en validación no mejora
early_stop_transfer = EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    restore_best_weights=True
)

In [ ]:
# Entrenar solo la cabeza personalizada (las capas de EfficientNet están congeladas)
history_transfer = model_transfer.fit(
    x=X_train_raw,
    y=y_train,
    validation_data=(X_test_raw, y_test),
    callbacks=[early_stop_transfer],
    batch_size=64,   # Batch menor porque EfficientNet consume más memoria GPU
    epochs=50
)

#### Evaluación del Modelo con Transfer Learning

In [ ]:
# Convertir el historial de entrenamiento a DataFrame para graficar
history_df_transfer = pd.DataFrame(history_transfer.history)

In [ ]:
# Graficar la evolución de la accuracy en entrenamiento y validación
history_df_transfer[['accuracy', 'val_accuracy']].plot();

In [ ]:
# Graficar la evolución de la pérdida en entrenamiento y validación
history_df_transfer[['loss', 'val_loss']].plot();

In [ ]:
# Evaluar el modelo de transfer learning en el conjunto de prueba
loss, accuracy = model_transfer.evaluate(X_test_raw, y_test, verbose=0)
print(f'Loss en test: {loss:.4f}')
print(f'Accuracy en test: {accuracy:.4f}')

In [ ]:
# Predecir clases con el modelo de transfer learning (datos en [0, 255])
predictions_transfer = np.argmax(model_transfer.predict(X_test_raw), axis=1)

In [ ]:
# Reporte de clasificación por clase
print(classification_report(y_test, predictions_transfer, target_names=class_names))

In [ ]:
# Matriz de confusión del modelo con transfer learning
plt.figure(figsize=(10, 8))
sns.heatmap(
    confusion_matrix(y_test, predictions_transfer), 
    annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names
)
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.title('Matriz de Confusión - EfficientNetB0 Transfer Learning');

### Predicción con Imagen Individual

In [ ]:
# Seleccionar una imagen individual del conjunto de prueba
idx = 100

# Para visualización usamos la versión normalizada [0, 1]
my_image_norm = X_test_norm[idx]
my_image_raw = X_test_raw[idx]

# Visualizar la imagen con su etiqueta real
plt.imshow(my_image_norm)
plt.title(f'Etiqueta real: {class_names[y_test[idx]]}')
plt.axis('off');

In [ ]:
# Predicción con el modelo custom (espera entrada normalizada de 32x32)
pred_custom = np.argmax(model_custom.predict(np.expand_dims(my_image_norm, axis=0)), axis=1)
print(f'CNN Custom predice: {class_names[pred_custom[0]]}')

In [ ]:
# Predicción con el modelo de transfer learning (espera rango [0, 255]; internamente redimensiona a 224x224)
pred_transfer = np.argmax(model_transfer.predict(np.expand_dims(my_image_raw, axis=0)), axis=1)
print(f'EfficientNetB0 predice: {class_names[pred_transfer[0]]}')